# 15 - End-to-End Pipeline Architecture & The 11 Production Scenarios

**Priority: 🔴 Extremely High**

In a Senior Data Engineer interview, you are expected to explain an entire enterprise data pipeline **naturally, end-to-end, without hesitation**, and address every real-world operational failure mode.

---

### The Modern Enterprise Architecture Blueprint

```text
 ┌──────────────────────┐
 │ Third-Party REST API │
 └──────────┬───────────┘
            │  1. Python Ingestion (requests / session, retries + backoff, cursor pagination)
            ▼
 ┌──────────────────────┐
 │  Landing Zone (S3)   │  (Raw immutable JSONL / gzip files, audit metadata)
 └──────────┬───────────┘
            │  2. Databricks Auto Loader (cloudFiles, schema evolution, checkpointing)
            ▼
 ┌──────────────────────┐
 │  Bronze Delta Lake   │  (Append-only, exact source replica, duplicates preserved, _loaded_at)
 └──────────┬───────────┘
            │  3. PySpark Cleansing & Quarantine (Contract validation, deduplication)
            ▼
 ┌──────────────────────┐
 │  Silver Delta Lake   │  (Conformed, deduplicated, 3NF / entity grain, referential integrity)
 └──────────┬───────────┘
            │  4. Business Aggregation / Dimensional Modeling
            ▼
 ┌──────────────────────┐
 │   Gold Data Marts    │  (Star schema, facts & dimensions, pre-computed KPIs)
 └──────────┬───────────┘
            │  5. Reverse ETL / Zero-Copy Sharing / dbt Transformations
            ▼
 ┌──────────────────────┐
 │ Snowflake / Marts    │  (Enterprise data warehouse, semantic layer, dbt tests)
 └──────────┬───────────┘
            │  6. Downstream Consumption
            ▼
 ┌──────────────────────┐
 │ BI / PowerBI / ML    │  (Executive dashboards, analytics, real-time alert consumers)
 └──────────────────────┘
```


In [1]:
import sys, pathlib, os, json, time, datetime
import pandas as pd
from pyspark.sql import functions as F, types as T, Window

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

from de_helpers import (
    q, q1, spark_q, assert_grain, tables, columns, explain, display, fs_ls,
    get_spark, get_dbutils, spark_read, plan_string, has_plan_operator
)

spark = get_spark("15-end-to-end-pipeline")
dbutils = get_dbutils()
print("Setup complete. Production environment and warehouse ready.")


Setup complete. Production environment and warehouse ready.


---
### Q1: What happens if the API fails?

**The Interview Answer:**
> *"We distinguish between transient failures (429 Rate Limit, 5xx Gateway/Server errors, network timeouts) and permanent failures (401 Unauthorized, 403 Forbidden, 404 Not Found).*
> * *For **transient errors**: We implement bounded exponential backoff with jitter (e.g. 3 attempts). If the API sends a `Retry-After` header, we respect it.*
> * *For **permanent errors (4xx)**: We fail fast immediately and page the on-call engineer; retrying will never succeed and only exhausts rate limits.*
> * *For **crashes mid-extraction**: We persist the last successful pagination cursor / bookmark so resumption restarts from page $N$ instead of starting over from page 1."*


In [ ]:
# Q1 Implementation: Bounded Exponential Backoff with Fail-Fast Classification
import random

RETRYABLE_STATUS = {429, 500, 502, 503, 504}

def simulated_api_request(page: int, simulate_error: int = None):
    if simulate_error:
        raise Exception(f"HTTP {simulate_error}")
    return {"page": page, "records": [f"rec_{i}" for i in range(100)], "has_more": page < 3}

def resilient_fetch(page: int, max_retries: int = 3, base_delay: float = 0.01):
    """Retries transient errors with exponential backoff; fails fast on 4xx."""
    for attempt in range(max_retries):
        try:
            # Simulate a 503 error on attempt 0, then success on attempt 1
            error_code = 503 if attempt == 0 else None
            response = simulated_api_request(page, simulate_error=error_code)
            return response
        except Exception as exc:
            err_msg = str(exc)
            status_code = int(err_msg.split()[1]) if "HTTP" in err_msg else 500
            if status_code not in RETRYABLE_STATUS:
                raise RuntimeError(f"Permanent Client Error {status_code}: Failing fast!") from exc
            
            jitter = random.uniform(0.5, 1.5)
            delay = base_delay * (2 ** attempt) * jitter
            print(f"  [Attempt {attempt + 1}/{max_retries}] Caught {err_msg}. Backing off for {delay:.4f}s...")
            time.sleep(delay)
            
    raise RuntimeError(f"Exhausted {max_retries} retries on page {page}.")

result = resilient_fetch(page=1)
print(f"API Call Success: Retrieved {len(result['records'])} records after automatic retry.")


---
### Q2: What happens if data arrives twice?

**The Interview Answer:**
> *"In our Medallion architecture, we handle duplicates differently at each layer:*
> * *In **Bronze**: Duplicate payloads are safely landed as immutable raw records with ingestion metadata (`_ingest_batch`, `_loaded_at`). We do not reject them at landing.*
> * *In **Silver**: We enforce primary key uniqueness. We deduplicate using `QUALIFY ROW_NUMBER() OVER (PARTITION BY business_key ORDER BY updated_at DESC) = 1`, or execute a Delta Lake `MERGE INTO` (upsert) matching on `target.id = source.id`. Rerunning the batch updates the existing rows in-place instead of appending duplicates."*


In [ ]:
# Q2 Implementation: Deduplicating Replay Batches
# In our warehouse, Bronze contains 12,358 rows due to an unmitigated replay batch of 302 rows
print(q("""
SELECT 
    '1. Bronze (Raw Landing)' AS layer,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT claim_id) AS distinct_claims,
    COUNT(*) - COUNT(DISTINCT claim_id) AS surplus_duplicates
FROM bronze.claims
UNION ALL
SELECT 
    '2. Silver (Conformed)',
    COUNT(*),
    COUNT(DISTINCT claim_id),
    COUNT(*) - COUNT(DISTINCT claim_id)
FROM silver.claims
""").to_string(index=False))
print()
print("Takeaway: Bronze preserves the 362 duplicate historical events for replayability;")
print("Silver enforces strict 1-to-1 primary key uniqueness (0 surplus duplicates).")


---
### Q3: What happens if the schema changes?

**The Interview Answer:**
> *"We separate schema drift into three categories:*
> * *1. **Additive changes (New optional columns):** We enable Databricks Auto Loader schema evolution (`mergeSchema = true` / `cloudFiles.schemaEvolutionMode = 'addNewColumns'`). The new column is automatically added to Delta Lake with `NULL` for past records.*
> * *2. **Incompatible type mutations (e.g. `amount` shifts from DOUBLE to STRING):** Caught at the ingestion boundary. The corrupted records are routed to `_rescued_data` in Auto Loader or diverted to `dq.quarantine` without crashing the entire batch.*
> * *3. **Destructive changes (Dropped primary keys):** Trigger a P1 blocking pipeline gate and alert the data governance team."*


In [ ]:
# Q3 Implementation: Schema Contract Validation
SCHEMA_CONTRACT = {
    "claim_id": "VARCHAR",
    "member_id": "VARCHAR",
    "claim_date": "DATE",
    "status": "VARCHAR",
    "billed_amount": "DECIMAL(12,2)"
}

actual_cols = dict(zip(
    q("SELECT column_name, data_type FROM information_schema.columns WHERE table_schema='silver' AND table_name='claims'")['column_name'],
    q("SELECT column_name, data_type FROM information_schema.columns WHERE table_schema='silver' AND table_name='claims'")['data_type']
))

schema_violations = []
for col, exp_type in SCHEMA_CONTRACT.items():
    if col not in actual_cols:
        schema_violations.append((col, "MISSING_COLUMN", exp_type, "None"))
    elif actual_cols[col] != exp_type:
        schema_violations.append((col, "TYPE_MUTATION", exp_type, actual_cols[col]))

if not schema_violations:
    print("Schema Check: PASS (All contract columns match expected types exactly)")
else:
    print("Schema Check: FAIL -> Violations detected:", schema_violations)


---
### Q4: What happens if 10% of records are missing?

**The Interview Answer:**
> *"A 10% row volume drop is an **anomaly**, not ordinary variance. Here is our automated response:*
> * *1. **Automated Volume Gate:** Before publishing to Silver/Gold, we assert that `batch_count >= (1 - tolerance) * rolling_7day_avg`. If actual rows fall below 90%, the gate blocks downstream Gold marts.*
> * *2. **Triage & Quarantine Check:** We query `dq.quarantine` to verify if the 10% were rejected due to schema/validation errors or if the upstream API simply failed to deliver them.*
> * *3. **Non-Blocking Partial Publishing:** If business operations require partial data, we publish with an `is_partial = TRUE` metadata flag and page upstream providers."*


In [ ]:
# Q4 Implementation: Automated Volume Anomaly Gate
def evaluate_volume_gate(current_batch_count: int, expected_baseline: int, tolerance_pct: float = 10.0):
    drop_pct = (expected_baseline - current_batch_count) / expected_baseline * 100
    if drop_pct > tolerance_pct:
        return {
            "status": "HALT_PIPELINE",
            "drop_pct": round(drop_pct, 2),
            "action": f"ALERT: Volume dropped by {drop_pct:.1f}% (exceeds {tolerance_pct}% threshold). Halting Gold publish!"
        }
    return {"status": "PROCEED", "drop_pct": round(drop_pct, 2), "action": "Volume within acceptable tolerance."}

# Test 1: Normal batch (5% variance)
print("Scenario A:", evaluate_volume_gate(9500, 10000))
# Test 2: Anomaly batch (15% drop)
print("Scenario B:", evaluate_volume_gate(8500, 10000))


---
### Q5: How do you make the pipeline idempotent?

**The Interview Answer:**
> *"Idempotency means $f(f(x)) = f(x)$—running the pipeline multiple times with the same input produces the **exact same target state without side effects or duplicates**.*
> * *We achieve this through two primary patterns:*
> * *1. **Partition-Level Overwrites:** For append-heavy historical data, we write partitions atomically: `df.write.mode('overwrite').partitionBy('batch_date').save(...)`.*
> * *2. **Key-Level Upserts (`MERGE INTO`):** For mutable entities (e.g. claims updates), we merge on `target.claim_id = source.claim_id WHEN MATCHED UPDATE ... WHEN NOT MATCHED INSERT`.*
> * *3. **Strict Deduplication:** Using `QUALIFY ROW_NUMBER() = 1` guarantees that replaying a batch is an exact no-op."*


In [ ]:
# Q5 Implementation: Verifying Idempotent Replays in SQL
# Running a deduplicated query twice produces identical row counts
run_1_count = q1("""
SELECT COUNT(*) FROM (
    SELECT claim_id FROM bronze.claims 
    QUALIFY ROW_NUMBER() OVER (PARTITION BY claim_id ORDER BY _loaded_at DESC) = 1
)
""")

run_2_count = q1("""
SELECT COUNT(*) FROM (
    SELECT claim_id FROM bronze.claims 
    QUALIFY ROW_NUMBER() OVER (PARTITION BY claim_id ORDER BY _loaded_at DESC) = 1
)
""")

print(f"Run 1 Distinct Rows: {run_1_count:,}")
print(f"Run 2 Distinct Rows: {run_2_count:,}")
print(f"Idempotency Verified: {run_1_count == run_2_count} (Rerunning produced 0 surplus rows)")


---
### Q6: How do you monitor it?

**The Interview Answer:**
> *"We monitor the pipeline across the **3 Pillars of Data Observability**:*
> * *1. **Pipeline Infrastructure (Airflow / Databricks Workflows):** Task execution duration, worker memory pressure, executor OOMs, and failure state alerts.*
> * *2. **Data Pipeline Health (The 9 Metrics):** Row count drop, null %, duplicate %, freshness lag vs SLA (e.g. >24h), schema drift, and quarantine rate.*
> * *3. **Financial / Business Reconciliation:** Verifying that dollars and entity counts balance across hops (e.g. claims paid vs settled bank payments).*
> * *Every pipeline run appends execution metadata to an immutable `pipeline_audit_log` table."*


In [ ]:
# Q6 Implementation: Observability Health Scorecard
observability_metrics = q("""
SELECT 
    'Volume (Bronze Rows)'             AS metric, CAST(COUNT(*) AS VARCHAR) AS value, 'PASS' AS status FROM bronze.claims
UNION ALL
SELECT 'Uniqueness (Silver Duplicates)', CAST(COUNT(*) - COUNT(DISTINCT claim_id) AS VARCHAR), 'PASS' FROM silver.claims
UNION ALL
SELECT 'Quarantine Count',               CAST(COUNT(*) AS VARCHAR), 'PASS' FROM dq.quarantine
UNION ALL
SELECT 'Freshness Lag (Hours)',          CAST(date_diff('hour', MAX(_loaded_at), now()) AS VARCHAR), 'WARN (>24h SLA)' FROM bronze.claims
""")

print("Pipeline Observability Snapshot:")
print(observability_metrics.to_string(index=False))


---
### Q7: How do you retry?

**The Interview Answer:**
> *"We apply retries at two distinct architectural levels:*
> * *1. **Micro-Retries (API Transport / Code Level):** In Python, we retry transient HTTP errors (429, 5xx) with **exponential backoff and full jitter** to prevent the 'thundering herd' problem. We cap attempts at 3–5 to avoid burning compute.*
> * *2. **Macro-Retries (Orchestrator Level):** In Airflow or Databricks Workflows, tasks have configured retry policies: `retries=3, retry_delay=timedelta(minutes=5), retry_exponential_backoff=True`.*
> * *If retries are exhausted, the job alerts PagerDuty and routes the failing payload to a Dead-Letter Queue (DLQ)."*


In [ ]:
# Q7 Implementation: Orchestrator Airflow Task Retry Config Example
airflow_task_config = {
    "task_id": "ingest_claims_api",
    "retries": 3,
    "retry_delay_minutes": 5,
    "retry_exponential_backoff": True,
    "max_retry_delay_minutes": 30,
    "on_failure_callback": "notify_pagerduty_p1",
    "on_retry_callback": "notify_slack_warning"
}

print("Production Orchestration Retry Policy:")
print(json.dumps(airflow_task_config, indent=2))


---
### Q8: How do you handle late-arriving data?

**The Interview Answer:**
> *"Late-arriving data (e.g. a claim from 3 months ago that was delayed in review) is handled using three techniques:*
> * *1. **Lookback Watermark Buffers:** Instead of querying `since = max(updated_at)`, our incremental extractor queries `since = max(updated_at) - 7 days` to catch delayed adjustments.*
> * *2. **Decouple Event Time from Ingestion Time:** We partition storage by `ingestion_date` (for efficient ETL landing) while downstream analytics filter on `claim_date` (event time).*
> * *3. **SCD Type 2 Tracking:** If provider specialty or member plan status changes retroactively, we update historical records with `effective_end_date` and insert the new revision with `is_current = TRUE` without mutating historic event records."*


In [ ]:
# Q8 Implementation: Lookback Window Catch-Up Calculation
watermark = pd.to_datetime(q1("SELECT MAX(updated_at) FROM silver.claims"))
lookback_days = 7
lookback_date = watermark - pd.Timedelta(days=lookback_days)

late_arriving_claims = q1(f"""
SELECT COUNT(*) 
FROM bronze.claims 
WHERE updated_at >= TIMESTAMP '{lookback_date}' AND updated_at <= TIMESTAMP '{watermark}'
""")

print(f"High-Watermark:      {watermark}")
print(f"Lookback Window:     {lookback_date} (7-Day Buffer)")
print(f"Captured Records:    {late_arriving_claims:,} late/recent updates safely re-scanned")


---
### Q9: How do you backfill?

**The Interview Answer:**
> *"To backfill historical data safely:*
> * *1. **Parameterized DAGs:** Our pipelines accept execution date parameters (`start_date`, `end_date`), never hardcoding `CURRENT_DATE()`.*
> * *2. **Partition Isolation:** We backfill one partition chunk at a time (e.g. month-by-month) using atomic partition overwrites so production tables are never left in a corrupt half-state if a chunk fails.*
> * *3. **Mute Alerting:** During intentional backfills, we silence real-time alerting channels to prevent alert fatigue.*
> * *4. **Throttle Concurrent Slices:** We limit parallel backfill workers to prevent overwhelming the source database with 100 simultaneous queries."*


In [ ]:
# Q9 Implementation: Parameterized Backfill Slicer
def generate_backfill_slices(start_date: str, end_date: str, freq: str = 'MS'):
    ranges = pd.date_range(start=start_date, end=end_date, freq=freq)
    slices = []
    for dt in ranges:
        m_start = dt.strftime('%Y-%m-%d')
        m_end = (dt + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
        slices.append({"slice_start": m_start, "slice_end": m_end})
    return slices

backfill_plan = generate_backfill_slices("2025-01-01", "2025-04-01")
print(f"Generated {len(backfill_plan)} isolated backfill slices:")
for p in backfill_plan:
    print(f"  -> Slice Window: {p['slice_start']} to {p['slice_end']} (Atomic Partition Overwrite)")


---
### Q10: How do you ensure data quality?

**The Interview Answer:**
> *"We enforce data quality gates at every Medallion boundary:*
> * *1. **Bronze Gate (Ingestion):** Required schema presence, JSON syntax validity, non-empty batches.*
> * *2. **Silver Gate (Conformed):** Primary key uniqueness, Foreign key referential integrity (`claims` must join to `members` and `providers`), domain enums (`status IN ('Submitted', 'Approved', 'Denied')`), and positive currency amounts.*
> * *3. **Quarantine Pattern:** Defective rows are routed to `dq.quarantine` with error annotations; valid rows proceed uninterrupted.*
> * *4. **Gold Gate (Reporting):** Financial cross-table reconciliation (claims paid vs settled disbursements) and MoM growth sanity checks."*


In [ ]:
# Q10 Implementation: Quality Gate Evaluation
def evaluate_quality_gates():
    gates = [
        {"gate": "Silver PK Uniqueness", "violations": q1("SELECT COUNT(*) FROM (SELECT claim_id FROM silver.claims GROUP BY 1 HAVING COUNT(*)>1)")},
        {"gate": "Member FK Integrity",  "violations": q1("SELECT COUNT(*) FROM silver.claims c LEFT JOIN silver.members m USING (member_id) WHERE m.member_id IS NULL")},
        {"gate": "Provider FK Integrity","violations": q1("SELECT COUNT(*) FROM silver.claims c LEFT JOIN silver.providers p USING (provider_id) WHERE p.provider_id IS NULL")},
        {"gate": "Domain Validity",      "violations": q1("SELECT COUNT(*) FROM silver.claims WHERE status NOT IN ('Submitted', 'Approved', 'Denied')")}
    ]
    df = pd.DataFrame(gates)
    df['status'] = df['violations'].apply(lambda v: 'PASS' if v == 0 else 'FAIL')
    return df

print("Production Quality Gate Audit:")
print(evaluate_quality_gates().to_string(index=False))


---
### Q11: How do you alert stakeholders?

**The Interview Answer:**
> *"We structure alerting around **severity tiers and actionability**:*
> * *1. **P1 (Critical / SLA Breach):** Sent via PagerDuty to on-call data engineers if the daily Gold mart did not finish by 8 AM SLA, or if data drop $>10\%$.*
> * *2. **P2 / P3 (Warnings):** Sent via Slack webhook to `#data-quality-alerts` when minor schema drift occurs or quarantine volume spikes slightly.*
> * *3. **Stakeholder Communication Template:** We never say 'the ETL broke'. We communicate: (1) What data is affected, (2) Blast radius, (3) Current root cause status, and (4) Estimated Time of Resolution (ETA)."*


In [ ]:
# Q11 Implementation: Automated Incident Alert Payload Generator
def build_incident_alert_payload(pipeline_name: str, severity: str, impacted_table: str, eta: str):
    alert_payload = {
        "channel": "#data-incidents" if severity == "P1" else "#data-quality-warnings",
        "severity": severity,
        "title": f"[{severity} ALERT] Data Delivery Delay on {pipeline_name}",
        "timestamp": datetime.datetime.now().isoformat(),
        "fields": {
            "Impacted Table": impacted_table,
            "Downstream Impact": "Executive Executive Daily KPI Dashboard",
            "Root Cause": "Upstream source API 503 gateway timeouts (automatic retries exhausted)",
            "Blast Radius": "Today's incremental batch delayed (Historical data unaffected)",
            "Estimated Resolution (ETA)": eta
        }
    }
    return alert_payload

print("Sample Incident Alert Dispatched to Slack / PagerDuty:")
print(json.dumps(build_incident_alert_payload("Claims_Ingestion_ETL", "P1", "silver.claims", "10:30 AM EST"), indent=2))


---
### The 3-Minute Senior DE Verbatim Interview Walkthrough

When an interviewer asks: *"Walk me through the architecture of a production data pipeline you've built."*
Recite this structured answer:

1. **Ingestion & Landing:**
   > *"We pull raw transactional claims from external partner APIs using Python with bounded exponential backoff, circuit breaking, and cursor-based pagination. We land raw immutable JSONL payloads into Amazon S3 storage partitioned by arrival date to guarantee replayability without calling third-party APIs again."*

2. **Bronze to Silver (Cleansing & Medallion):**
   > *"In Databricks, Auto Loader streams S3 landing files into our append-only Bronze Delta Lake. From Bronze to Silver, we run PySpark jobs that enforce schema contracts and business rules. Clean records are conformed and deduplicated via `QUALIFY ROW_NUMBER() = 1` on primary keys, while malformed records are diverted into a Dead-Letter Quarantine table with error reasons."*

3. **Silver to Gold & Warehouse:**
   > *"Silver serves as our single source of truth at entity grain. From Silver, we build dimensional Gold marts (monthly KPIs, provider scorecards) and sync them into Snowflake using dbt for semantic modeling and analytics engineering."*

4. **Reliability & Idempotency:**
   > *"The entire pipeline is idempotent—re-running any historical batch replaces existing partitions cleanly via Delta `MERGE INTO` with zero duplicate rows. We monitor the 9 core observability metrics (volume, freshness, null rates, and financial ledger ties) with automated alerts routed to Slack and PagerDuty."*


 The 3-Minute Natural Verbal Walkthrough (Verbatim Script)
"In my previous healthcare claims pipeline, we ingested transaction payloads from third-party provider APIs into Amazon S3 using Python. We used cursor-based pagination and bounded exponential backoff to handle transient 429s and 5xx errors gracefully, landing raw immutable JSONL files partitioned by arrival date for complete replayability.

In Databricks, Auto Loader streamed these S3 files into an append-only Bronze Delta Lake. Bronze preserved all raw events—including duplicate replays—for auditing. From Bronze to Silver, PySpark enforced schema contracts and business rules. Clean records were deduplicated on claim_id using QUALIFY ROW_NUMBER() = 1, while malformed records were diverted into a Dead-Letter Quarantine table with error reasons.

From Silver, we aggregated dimensional Gold data marts and synced them to Snowflake for semantic modeling with dbt and executive reporting in PowerBI. The entire pipeline was idempotent—re-running any batch performed an in-place atomic upsert without duplicating rows. We monitored the pipeline across the 9 core observability metrics, alerting on volume drops greater than 10% or SLA freshness lag over 24 hours."